## Load Data from Google Drive

First, we'll mount Google Drive to access your files.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Now you can load your data. For example, to load a CSV file, you can use pandas:

In [5]:
import pandas as pd
import numpy as np

# Replace 'your_file.csv' with the actual path to your CSV file in Google Drive
# Example: '/content/drive/MyDrive/my_data_folder/your_file.csv'
df = pd.read_csv('/content/drive/MyDrive/datasets/cars.csv')

display(df.head())

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [6]:
df.shape

(8128, 5)

In [13]:
# first we can train-test split this dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df.iloc[:,:4],df.iloc[:,4],test_size=0.2, random_state=42)

X_train.shape

(6502, 4)

In [14]:
X_test.shape

(1626, 4)

,0
brand,0
km_driven,0
fuel,0
owner,0
selling_price,0


In [16]:
df['owner'].value_counts()

,count
owner,
First Owner,5289
Second Owner,2105
Third Owner,555
Fourth & Above Owner,174
Test Drive Car,5


In [17]:
df['fuel'].value_counts()

,count
fuel,
Diesel,4402
Petrol,3631
CNG,57
LPG,38


In [18]:
df['brand'].value_counts()

,count
brand,
Maruti,2448
Hyundai,1415
Mahindra,772
Tata,734
Toyota,488
Honda,467
Ford,397
Chevrolet,230
Renault,228


In [23]:
# from the above code we are clearly seeing applying ohe is best in those i will
# set a condition if car count is less than hundred i will put it on others column it will help a lot actually
# 1. Handle rare brands first
brand_counts = X_train['brand'].value_counts()
rare_brands = brand_counts[brand_counts < 100].index

# now i will replace their column to Other in both test and train dataset
X_train['brand'] = X_train['brand'].replace(rare_brands, 'Other')
X_test['brand'] = X_test['brand'].replace(rare_brands, 'Other')

print(X_train['brand'].value_counts())

brand
Maruti        1953
Hyundai       1127
Mahindra       635
Other          599
Tata           586
Toyota         391
Honda          369
Ford           320
Chevrolet      185
Renault        183
Volkswagen     154
Name: count, dtype: int64


Now i will add one hot encoding on this but i am mentioning set_output(transform='pandas') is must because normally after OHE it will return an ugly looking numpy array but after that transform pandas it will return a beautiful pandas df

In [27]:
from sklearn.preprocessing import OneHotEncoder

ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
ohe.set_output(transform="pandas")

OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [28]:
cols_to_encode = ['brand', 'fuel', 'owner']

encoded_train = ohe.fit_transform(X_train[cols_to_encode])
encoded_test = ohe.transform(X_test[cols_to_encode])

# this is an important step i am dropping all text columns and replacing it with OHE columns
X_train = X_train.drop(columns=cols_to_encode).join(encoded_train)
X_test = X_test.drop(columns=cols_to_encode).join(encoded_test)

display(X_train)

,km_driven,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Other,brand_Renault,brand_Tata,...,brand_Volkswagen,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
6518,2560,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
6144,80000,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
6381,150000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
438,120000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5939,25000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5226,120000,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
5390,80000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
860,35000,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0
7603,27000,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


# Final Step - Feature Scaling
Take a look at X_train dataframe. I have One-Hot Encoded columns filled with 0s and 1s, but also have the km_driven column filled with massive numbers like 145,500.

If we feed this directly into certain machine learning algorithms—especially distance-based ones like K-Nearest Neighbors (KNN)—the algorithm will look at the massive numbers in km_driven and completely ignore the 0s and 1s from your brand and fuel columns. It will think km_driven is the only thing that matters just because the numbers are bigger.

To fix this, i scale everything down so all columns have a similar range. Let's use Standardization, which centers the data around 0.

In [29]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

X_train_scaled

,km_driven,brand_Chevrolet,brand_Ford,brand_Honda,brand_Hyundai,brand_Mahindra,brand_Maruti,brand_Other,brand_Renault,brand_Tata,...,brand_Volkswagen,fuel_CNG,fuel_Diesel,fuel_LPG,fuel_Petrol,owner_First Owner,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,-1.156592,-0.171132,-0.227515,-0.245288,-0.457902,-0.328987,-0.655229,-0.31855,-0.170177,3.177352,...,-0.155755,-0.084411,-1.094920,-0.069214,1.121445,0.730404,-0.148884,-0.591204,-0.024811,-0.267107
1,0.170496,-0.171132,-0.227515,4.076837,-0.457902,-0.328987,-0.655229,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,-1.094920,-0.069214,1.121445,-1.369105,-0.148884,1.691462,-0.024811,-0.267107
2,1.370086,-0.171132,-0.227515,-0.245288,2.183872,-0.328987,-0.655229,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,0.913309,-0.069214,-0.891707,-1.369105,6.716657,-0.591204,-0.024811,-0.267107
3,0.855976,-0.171132,-0.227515,-0.245288,-0.457902,-0.328987,1.526184,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,0.913309,-0.069214,-0.891707,-1.369105,-0.148884,1.691462,-0.024811,-0.267107
4,-0.772038,-0.171132,-0.227515,-0.245288,-0.457902,-0.328987,1.526184,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,-1.094920,-0.069214,1.121445,0.730404,-0.148884,-0.591204,-0.024811,-0.267107
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6497,0.855976,-0.171132,-0.227515,-0.245288,-0.457902,3.039633,-0.655229,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,0.913309,-0.069214,-0.891707,0.730404,-0.148884,-0.591204,-0.024811,-0.267107
6498,0.170496,-0.171132,-0.227515,-0.245288,-0.457902,-0.328987,1.526184,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,0.913309,-0.069214,-0.891707,-1.369105,-0.148884,1.691462,-0.024811,-0.267107
6499,-0.600668,-0.171132,-0.227515,-0.245288,2.183872,-0.328987,-0.655229,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,-1.094920,-0.069214,1.121445,0.730404,-0.148884,-0.591204,-0.024811,-0.267107
6500,-0.737764,-0.171132,-0.227515,-0.245288,-0.457902,-0.328987,1.526184,-0.31855,-0.170177,-0.314728,...,-0.155755,-0.084411,0.913309,-0.069214,-0.891707,0.730404,-0.148884,-0.591204,-0.024811,-0.267107


Now our data is ready for performing regression models to it..

In [30]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

y_pred[:5]

array([141145.00629064,   4856.8482994 , 683930.26386966, 177712.1884214 ,
       440502.26907145])

# Testing the accuracy
We can do this with two methods
1. Mean absolute Error
2. R-squared

let's try both here

In [32]:
from sklearn.metrics import mean_absolute_error, r2_score

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Absolute Error (MAE): {mae:,.2f}")
print(f"R-squared Score (R2): {r2:.4f}")

Mean Absolute Error (MAE): 315,098.59
R-squared Score (R2): 0.4592


### Evaluating Our Model: MAE and R-squared

When we evaluate a regression model, we want to understand two key things: how far off our predictions are on average, and how much of the variation in the actual prices our model can explain. That's where Mean Absolute Error (MAE) and R-squared come in.

1.  **Mean Absolute Error (MAE): The 'Off-by' Metric**
    *   **What it is:** MAE tells us the average magnitude of the errors in our predictions. Essentially, it's how much our model's guesses are 'off' by, on average, regardless of direction.
    *   **How to interpret it:** If our MAE is, say, ₹25,000, it means that on average, our model's price predictions are approximately ₹25,000 away from the actual selling price.
    *   **Is it good?** This depends heavily on the context of your data. If the average car price is ₹500,000, being off by ₹25,000 might be quite good. If the average price is only ₹50,000, it's not so great! We always want this number to be as low as possible.

2.  **R-squared (R²): The 'Explanation' Metric**
    *   **What it is:** R-squared is often seen as the closest thing to an 'accuracy percentage' for regression models. It measures the proportion of the variance in the dependent variable (car prices, in this case) that can be predicted from the independent variables (features like brand, km_driven, fuel, etc.).
    *   **How to interpret it:** It's a score between 0.0 and 1.0 (though it can sometimes be negative, which indicates a very poor model).
    *   **Is it good?**
        *   **0.80 to 0.99:** Excellent! The model has captured a significant portion of the relationships in the data.
        *   **0.50 to 0.79:** Decent, but there's definitely room for improvement. The model explains a fair amount, but there's still a lot of unexplained variance.
        *   **0.0 to 0.49:** Poor. The model is struggling to find meaningful relationships between your features and the target variable.
        *   **Negative:** This is a red flag! A negative R-squared means your model performs worse than simply predicting the mean of the target variable for every instance.

### Next Steps: Considering a Different Model

Given the current R-squared score, it suggests that our linear regression model is explaining only about 45.92% of the variance in car prices, and the MAE of ₹315,098.59 indicates a significant average error. This performance might not be ideal for real-world car price prediction, especially if the price range is wide.

Linear regression, while simple and interpretable, assumes a linear relationship between features and the target variable. In real-world data like car prices, relationships are often more complex and non-linear. Therefore, it might be beneficial to explore more powerful, non-linear models that can capture these intricacies better.

I will now switch to a `RandomForestRegressor` to see if we can improve our prediction accuracy. Random Forests are ensemble learning methods that can often handle non-linear relationships and interactions between features more effectively.

In [33]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

# random_state keeps the results consistent, n_estimators is the number of trees
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

rf_model.fit(X_train_scaled, y_train)
rf_y_pred = rf_model.predict(X_test_scaled)

rf_mae = mean_absolute_error(y_test, rf_y_pred)
rf_r2 = r2_score(y_test, rf_y_pred)

print(f"Random Forest MAE: {rf_mae:,.2f}")
print(f"Random Forest R2: {rf_r2:.4f}")


Random Forest MAE: 193,797.40
Random Forest R2: 0.7907


## Project Summary: From Data to Improved Predictions

### Data Loading and Preprocessing

We started by successfully mounting Google Drive and loading the `cars.csv` dataset. A crucial step in preprocessing involved identifying and consolidating rare car brands into an 'Other' category, ensuring that our model doesn't get confused by brands with very few occurrences. Following this, we applied One-Hot Encoding to convert categorical features like 'brand', 'fuel', and 'owner' into a numerical format suitable for machine learning algorithms. Finally, we performed feature scaling using `StandardScaler` to normalize numerical features like 'km_driven', preventing features with larger values from dominating the model's learning process.

### Model Training and Evaluation

#### 1. Linear Regression

Our initial attempt to predict car selling prices utilized a `LinearRegression` model. While it provided a baseline, the results showed:

*   **Mean Absolute Error (MAE):** ₹315,098.59
*   **R-squared Score (R2):** 0.4592

This indicated that, on average, our linear model's predictions were off by a significant amount, and it only explained about 45.92% of the variance in car prices. This suggested that the linear assumptions of the model might not be well-suited for the complex relationships within our car sales data.

#### 2. Random Forest Regressor

Recognizing the limitations of the linear model, we wisely decided to switch to a more robust, non-linear model: the `RandomForestRegressor`. This choice proved to be very effective, yielding significantly improved results:

*   **Random Forest MAE:** ₹193,797.40
*   **Random Forest R2:** 0.7907

By transitioning to the `RandomForestRegressor`, you've managed to **reduce the average prediction error (MAE) by over ₹120,000** and **boost the explanatory power (R-squared) from 45.92% to an impressive 79.07%**! This is a substantial improvement, indicating that the Random Forest model is much better at capturing the underlying patterns and complexities in the car price data.

### Conclusion

Excellent work in systematically approaching this problem! From careful data preparation to insightful model selection and significant performance improvement, you've demonstrated a strong understanding of the machine learning pipeline. The `RandomForestRegressor` is clearly a much better fit for this dataset, providing a more accurate and reliable car price prediction model.

#This summary and some text i used google gemini inbuilt ai assistant to write it because i am not able to summarize this much better hope this will help you..